In [1]:
import pandas as pd
from pathlib import Path

### Load dataset, pre-process, and filter shortlisted NMIs

In [2]:
filename = Path.cwd().parent / "Data Science Project" / "LMS_2013-01-01_2026-03-24_HALF_HOUR_au.pq"
df = pd.read_parquet(filename, engine='fastparquet')
df.columns = df.columns.str.replace(" consumption", "")

# Remove NMIs 6102507141 and VAAA003225
print("Original number of NMIs: " + str(len(df.columns)-1))
df.drop(columns=["6102507141","VAAA003225"], inplace=True)
print("New number of NMIs: "  + str(len(df.columns)-1))

df['date'] = pd.to_datetime(df['date'])
df.sort_values(by='date', ascending=True)
print(f"Duplicated period: {df[df['date'].duplicated()]['date']}")

df = df.set_index('date')

Original number of NMIs: 101
New number of NMIs: 99
Duplicated period: Series([], Name: date, dtype: datetime64[us])


In [3]:
filename = Path.cwd().parent / "Data Science Project" / "NMI to weather station.xlsx"
weathermap_df = pd.read_excel(filename, usecols=['NMI','Station Number'], dtype={'Station Number': str})
weathermap_df = weathermap_df.dropna().drop_duplicates().reset_index(drop=True)

weathermap_df

,NMI,Station Number
0,6102002302,086338
1,6102005454,086338
2,6102005592,086338
3,6102009742,086338
4,6102009743,086338
...,...,...
89,VAAA003176,086338
90,VAAA003194,086338
91,VAAA003429,086338
92,VAAA004066,086338


In [4]:
daily_df = df.resample("D").sum()
daily_df

,6102000812,6102002302,6102005454,6102005592,6102009742,6102009743,6102009744,6102023971,6102038376,6102046251,...,VAAA003100,VAAA003176,VAAA003194,VAAA003197,VAAA003429,VAAA004066,VCCCAE0035,VCCCBC0096,VCCCSC0045,VCCCSD0058
date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,974.544,618.735,1379.600,1269.519,220.580,1625.352,1386.580,10685.157,4140.360,1141.48,...,9097.671,1451.150,910.080,1345.859,0.000,0.000,5447.697,1197.300,1106.832,907.381
2013-01-02,1131.152,1027.506,2886.558,2550.559,235.501,2009.416,1735.242,11917.137,4579.280,1376.68,...,11204.940,1638.342,1788.140,3606.159,0.000,0.000,5778.165,1291.410,1266.064,1062.508
2013-01-03,1418.752,1292.942,3119.639,5567.840,227.034,2696.810,2428.361,12809.396,5278.599,1598.48,...,12904.244,1965.340,2460.460,4186.940,0.000,0.000,7253.330,1506.240,1602.872,1276.815
2013-01-04,1671.390,1748.925,3055.560,7525.639,221.600,3447.488,3187.360,13754.219,6414.080,1558.32,...,14078.202,2506.008,2816.840,3804.498,0.000,0.000,8547.730,1656.870,1971.728,1491.666
2013-01-05,1136.448,774.644,1990.200,2087.479,213.422,1131.610,897.344,11528.219,4944.840,1381.76,...,10305.364,1025.500,2919.016,1505.340,0.000,0.000,6221.261,1291.665,600.224,1476.144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-19,1205.056,925.480,3887.600,4250.700,355.360,3071.516,2617.500,21206.500,3398.460,1934.32,...,2598.980,2311.320,2319.120,6169.140,276.672,15.616,6783.392,627.420,1243.170,1713.472
2026-03-20,1071.232,607.040,3738.200,3637.200,359.360,3060.788,2590.980,20325.700,3194.580,1556.56,...,2310.710,1995.420,1980.840,5425.620,223.968,15.696,5694.752,590.880,1024.910,1201.312
2026-03-21,913.184,350.320,2986.700,2774.800,316.624,1851.580,1445.100,17789.900,2274.180,1604.00,...,1063.160,1316.040,1644.840,4612.020,98.624,15.560,5364.128,588.660,760.990,993.888


In [5]:
def read_weather(filepath, ws_number, value_col):
    filename = list(filepath.glob('*.csv'))[0]
    df = pd.read_csv(filename, low_memory=False)
    df['date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
    df = df.set_index('date')
    if value_col == 'Maximum temperature (Degree C)':
        df.rename(columns={value_col: 'maxtemp'}, inplace=True)
        return df['maxtemp']
    elif value_col == 'Minimum temperature (Degree C)':
        df.rename(columns={value_col: 'mintemp'}, inplace=True)
        return df['mintemp']
    elif value_col == 'Rainfall amount (millimetres)':
        df.rename(columns={value_col: 'rainfall'}, inplace=True)
        return df['rainfall']
    else:
        return None

In [6]:
weather_dfs = []
ws_list = weathermap_df['Station Number'].drop_duplicates()
full_dates = pd.date_range(start='2013-01-01', end=max(df.index), freq='D')

for ws_number in ws_list:
    ws_number = str(ws_number).zfill(6)
    max_folder = Path.cwd().parent / "Data Science Project" / "weather station" / ws_number / "MaxTemp"
    min_folder = Path.cwd().parent / "Data Science Project" / "weather station" / ws_number / "MinTemp"
    rain_folder = Path.cwd().parent / "Data Science Project" / "weather station" / ws_number / "Rainfall"

    maxtemp = read_weather(max_folder, ws_number, 'Maximum temperature (Degree C)')
    mintemp = read_weather(min_folder, ws_number, 'Minimum temperature (Degree C)')
    rainfall = read_weather(rain_folder, ws_number, 'Rainfall amount (millimetres)')
    
    station_weather = pd.concat([maxtemp, mintemp, rainfall],axis=1)
    station_weather = station_weather.reindex(full_dates)
    station_weather['Station Number'] = ws_number
    weather_dfs.append(station_weather)

ws_df = pd.concat(weather_dfs)
ws_df = ws_df.reset_index().rename(columns={'index': 'date'})
ws_df['date'] = pd.to_datetime(ws_df['date'])
ws_df['avgtemp'] = (ws_df['maxtemp'] + ws_df['mintemp']) / 2
ws_df = ws_df[['date', 'Station Number', 'maxtemp', 'mintemp', 'avgtemp', 'rainfall']]
ws_df

,date,Station Number,maxtemp,mintemp,avgtemp,rainfall
0,2013-01-01,086338,NaN,NaN,NaN,NaN
1,2013-01-02,086338,NaN,NaN,NaN,NaN
2,2013-01-03,086338,NaN,NaN,NaN,NaN
3,2013-01-04,086338,NaN,NaN,NaN,NaN
4,2013-01-05,086338,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
24145,2026-03-19,086038,19.6,14.8,17.20,1.2
24146,2026-03-20,086038,22.4,14.2,18.30,0.2
24147,2026-03-21,086038,23.4,13.0,18.20,0.0
24148,2026-03-22,086038,31.0,13.5,22.25,0.0


### Correlation

In [7]:
weathermap_df['NMI'] = weathermap_df['NMI'].astype(str)
weathermap_df['Station Number'] = weathermap_df['Station Number'].astype(str).str.zfill(6)

long_df = daily_df.reset_index().melt(id_vars='date', var_name='NMI', value_name='daily consumption')
long_df['NMI'] = long_df['NMI'].astype(str)
long_df = long_df.merge(weathermap_df[['NMI', 'Station Number']], on='NMI', how='left')
long_df

,date,NMI,daily consumption,Station Number
0,2013-01-01,6102000812,974.544,NaN
1,2013-01-02,6102000812,1131.152,NaN
2,2013-01-03,6102000812,1418.752,NaN
3,2013-01-04,6102000812,1671.390,NaN
4,2013-01-05,6102000812,1136.448,NaN
...,...,...,...,...
478165,2026-03-19,VCCCSD0058,1713.472,NaN
478166,2026-03-20,VCCCSD0058,1201.312,NaN
478167,2026-03-21,VCCCSD0058,993.888,NaN
478168,2026-03-22,VCCCSD0058,1028.544,NaN


In [8]:
results = []

# create weather lookup once
weather_lookup = {station: group[['date', 'maxtemp', 'mintemp', 'avgtemp', 'rainfall']] for station, group in ws_df.groupby('Station Number')}

# create NMI -> station lookup once
nmi_to_station = dict(zip(weathermap_df['NMI'].astype(str), weathermap_df['Station Number']))

for nmi in daily_df.columns:
    nmi_str = str(nmi)
    station = nmi_to_station.get(nmi_str)

    if station is None:
        continue

    weather = weather_lookup.get(station)

    if weather is None:
        continue

    merged = pd.DataFrame({'date': daily_df.index, 'consumption': daily_df[nmi].values}).merge(weather, on='date', how='left')

    results.append({
        'NMI': nmi_str,
        'Station Number': station,
        'maxtemp_corr': merged['consumption'].corr(merged['maxtemp']),
        'mintemp_corr': merged['consumption'].corr(merged['mintemp']),
        'avgtemp_corr': merged['consumption'].corr(merged['avgtemp']),
        'rainfall_corr': merged['consumption'].corr(merged['rainfall'])
    })

corr_df = pd.DataFrame(results)
corr_df
output_path = Path.cwd().parent / "Data Science Project" / "correlation_results.xlsx"
corr_df.to_excel(output_path, index=False)

In [9]:
corr_df['abs_temp_corr'] = corr_df['avgtemp_corr'].abs()
corr_df['temp_sensitivity'] = pd.cut(
    corr_df['abs_temp_corr'],
    bins=[0, 0.2, 0.4, 0.6, 1],
    labels=[
        'weak',
        'moderate',
        'strong',
        'very strong'
    ]
)
corr_df['temp_sensitivity'].value_counts()

temp_sensitivity
weak           67
moderate       16
very strong     6
strong          5
Name: count, dtype: int64

In [10]:
corr_df['abs_rain_corr'] = corr_df['rainfall_corr'].abs()
corr_df['rain_sensitivity'] = pd.cut(
    corr_df['abs_rain_corr'],
    bins=[0, 0.2, 0.4, 0.6, 1],
    labels=[
        'weak',
        'moderate',
        'strong',
        'very strong'
    ]
)
corr_df['rain_sensitivity'].value_counts()

rain_sensitivity
weak           94
moderate        0
strong          0
very strong     0
Name: count, dtype: int64

### Solar

In [11]:
solar_df = pd.read_excel(filename, usecols=['NMI','Solar (Y/N)'], dtype={'Solar (Y/N)': str})
solar_df['Solar (Y/N)'] = pd.to_numeric(solar_df['Solar (Y/N)'], errors='coerce')

# If one of the buildings has solar, assume NMI with solar
solar_df = solar_df.sort_values('Solar (Y/N)', ascending=False).drop_duplicates(subset='NMI').reset_index(drop=True)
solar_df

,NMI,Solar (Y/N)
0,VCCCSC0045,1.0
1,VAAA000175,1.0
2,6102005454,1.0
3,6103011168,1.0
4,6102332526,1.0
...,...,...
94,6203397519,NaN
95,VAAA003197,NaN
96,VCCCAE0035,NaN
97,VCCCBC0096,NaN


In [12]:
solar_lookup = dict(zip(solar_df['NMI'].astype(str),solar_df['Solar (Y/N)']))

solar_result = []

# daytime and overnight periods
midday_df = df.between_time('10:00', '15:00')
overnight_df = df.between_time('00:00', '05:00')

for nmi in df.columns:
    nmi_str = str(nmi)
    solar = solar_lookup.get(nmi_str)
    overall_mean = df[nmi].mean()
    midday_mean = midday_df[nmi].mean()
    overnight_mean = overnight_df[nmi].mean()

    # avoid divide-by-zero
    if overnight_mean == 0:
        midday_night_ratio = None
    else:
        midday_night_ratio = midday_mean / overnight_mean

    solar_result.append({
        'NMI': nmi_str,
        'solar': solar,
        'overall_mean': overall_mean,
        'midday_mean': midday_mean,
        'overnight_mean': overnight_mean,
        'midday_night_ratio': midday_night_ratio,
        'std_consumption': df[nmi].std(),
        'max_consumption': df[nmi].max()
    })

solar_stats = pd.DataFrame(solar_result)
solar_stats.groupby('solar')['midday_night_ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
solar,,,,,,,,
0.0,57.0,1.964782,0.905393,0.541366,1.290092,1.73039,2.378595,4.512291
1.0,35.0,1.778423,0.895754,0.527559,1.326700,1.53857,2.062997,4.930775
